Imports

In [1]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-28 14:22:46.556411: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-28 14:22:46.561209: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-28 14:22:46.561222: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [4]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=2:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

In [5]:
cluster.scale(jobs=1)

In [6]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [7]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat,retain_design_matricies=True)
test.extract_params(client)

In [114]:
def describe_parameters(client,model,dat,split):
    """
    Produces a convienient single-dataframe description of one split model. 
    Requires that you pass original data to compute cell-numbers
    """
    cell_counts=scm.get_cell_counts(client,dat,split=split)
    flattened_param=scm.flatten_param_representation(client,model,split=split)

In [ ]:
describe_parameters(client,model=test.by_cre_parameters.result(),dat=dat,split="cre_id")

SyntaxError: positional argument follows keyword argument (1687559622.py, line 1)

In [113]:
working=cell_counts.join(flattened_param)
working["r"]=np.exp(working["theta"])
working["sigmasquare"]=working["nb"]**2/working["r"]+working["nb"]
working["p"]=working["nb"]/working["sigmasquare"]
index_cols=working.columns.names
working=working.reset_index()
working

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
0,1,0,0,1,0,nobody,503,1.772204,0.454842,0.773563,2.167474,3.221222,0.550165
1,1,0,0,0,1,nobody,500,1.772204,0.880358,0.773563,2.167474,3.221222,0.550165
2,1,0,1,0,0,nobody,499,1.772204,0.772830,0.773563,2.167474,3.221222,0.550165
3,0,1,0,1,0,nobody,443,0.928970,0.454842,0.773563,2.167474,1.327123,0.699988
4,0,1,0,0,1,nobody,441,0.928970,0.880358,0.773563,2.167474,1.327123,0.699988
5,0,1,1,0,0,nobody,440,0.928970,0.772830,0.773563,2.167474,1.327123,0.699988
6,1,0,0,1,0,somebody,522,12.464120,0.519332,1.110927,3.037171,63.615099,0.195930
7,1,0,1,0,0,somebody,504,12.464120,0.799637,1.110927,3.037171,63.615099,0.195930
8,1,0,0,0,1,somebody,498,12.464120,0.902680,1.110927,3.037171,63.615099,0.195930
9,0,1,1,0,0,somebody,464,9.970093,0.799637,1.110927,3.037171,42.698820,0.233498


In [101]:
import dask.dataframe as dd

In [102]:
def auto_partition(pdf, target_mb_per_partition):
    """
    Convert a pandas DataFrame to a Dask DataFrame with automatic partition sizing.
    
    Parameters:
    - pdf: input pandas DataFrame
    - target_mb_per_partition: desired memory usage per partition (in megabytes)
    
    Returns:
    - ddf: Dask DataFrame with chosen number of partitions

    Minimum of 2!
    """
    est_bytes = pdf.memory_usage(index=True, deep=True).sum()
    target_bytes = target_mb_per_partition * 1_000_000
    npartitions = max(2, int(np.ceil(est_bytes / target_bytes)))
    return dd.from_pandas(pdf, npartitions=npartitions)

In [103]:
working["cre_id"]=working["cre_id"].astype("category")

In [104]:
working=auto_partition(working,50)

In [105]:
working

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
npartitions=2,,,,,,,,,,,,,
0,int64,int64,int64,int64,int64,category[known],int64,float64,float64,float32,float32,float64,float64
15,...,...,...,...,...,...,...,...,...,...,...,...,...
29,...,...,...,...,...,...,...,...,...,...,...,...,...


In [106]:
def explode_cells(df):
    return df.loc[df.index.repeat(df['cells'])].reset_index(drop=True)


working_exploded=working.map_partitions(explode_cells)
working_exploded=working_exploded.reset_index(drop=True)
#^key! Otherwise we get garbage index.

In [107]:
working_exploded

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
npartitions=2,,,,,,,,,,,,,
,int64,int64,int64,int64,int64,category[known],int64,float64,float64,float32,float32,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


In [108]:
import dask.array as da


In [109]:
r = working_exploded['r'].to_dask_array(lengths=True)
p = working_exploded['p'].to_dask_array(lengths=True)
nb_samples = da.random.negative_binomial(n=r, p=p, size=r.shape[0], chunks=r.chunks)


probabilities = working_exploded['zi'].to_dask_array(lengths=True)
bernoulli_trials = da.random.binomial(n=1, p=probabilities, size=probabilities.shape[0], chunks=probabilities.chunks)

zinb_samples=nb_samples * bernoulli_trials

working_exploded['zinb_sample'] = dd.from_dask_array(zinb_samples, columns='zinb_sample')

In [112]:
type(working_exploded.compute())

pandas.core.frame.DataFrame

In [57]:
cluster.close()